# PFE proposal D — Agentic maintenance assistant: tools, guardrails, evaluation harness
A tool-using agent (manual search, fault-code lookup, calculator, work-order creation) with guardrails:
tool allow-list, argument validation, step budget, **human approval for side effects**. Evaluated on a scenario suite.
CPU baseline planner = intent classifier + slot rules. **[GPU — Colab T4]** cell: an open LLM as ReAct planner.
The knowledge base is a small **synthetic** plant (fictional ACME assets) — replace with the host's procedures.

In [1]:
# Setup (Colab or local). CPU is enough unless a cell says [GPU — Colab T4].
%pip install -q scikit-learn pandas numpy rank_bm25

Note: you may need to restart the kernel to use updated packages.


## Package source (embedded from `code/src/`, identical to the tested files)

In [2]:
import os, sys
os.makedirs('src', exist_ok=True)
os.makedirs('src/agentkit', exist_ok=True)
os.makedirs('src/agentkit/resources', exist_ok=True)
sys.path.insert(0, os.path.abspath('src'))

In [3]:
%%writefile src/agentkit/__init__.py
"""Agentic maintenance assistant: tools + guardrails + rule/LLM planners + scenario evaluation."""
from . import agent, evaluate, planners, tools  # noqa: F401


Writing src/agentkit/__init__.py


In [4]:
%%writefile src/agentkit/agent.py
"""Agent loop with guardrails: tool allow-list, argument validation, step budget, human approval for side effects."""
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Any, Callable

from .tools import Tool


@dataclass
class Action:
    tool: str
    args: dict


@dataclass
class Final:
    answer: str
    sources: list[str] = field(default_factory=list)


@dataclass
class Step:
    action: Action
    observation: Any
    ok: bool


@dataclass
class Trace:
    query: str
    steps: list[Step] = field(default_factory=list)
    answer: str = ""
    sources: list[str] = field(default_factory=list)
    violations: list[str] = field(default_factory=list)

    @property
    def tools_called(self) -> list[str]:
        return [s.action.tool for s in self.steps]


class Agent:
    def __init__(self, planner, tools: dict[str, Tool], max_steps: int = 5,
                 approve: Callable[[str, dict], bool] = lambda tool, args: False):
        self.planner, self.tools, self.max_steps, self.approve = planner, tools, max_steps, approve

    def _validate(self, a: Action) -> str | None:
        if a.tool not in self.tools:
            return f"tool '{a.tool}' not allowed"
        spec = self.tools[a.tool].args
        missing = [k for k in spec if k not in a.args]
        extra = [k for k in a.args if k not in spec]
        bad = [k for k, t in spec.items() if k in a.args and not isinstance(a.args[k], t)]
        if missing or extra or bad:
            return f"invalid args for {a.tool}: missing={missing} extra={extra} wrong_type={bad}"
        return None

    def run(self, query: str) -> Trace:
        tr = Trace(query)
        for _ in range(self.max_steps):
            nxt = self.planner.next(query, tr)
            if isinstance(nxt, Final):
                tr.answer, tr.sources = nxt.answer, nxt.sources
                return tr
            err = self._validate(nxt)
            if err:
                tr.violations.append(err)
                tr.steps.append(Step(nxt, {"error": err}, False))
                continue
            tool = self.tools[nxt.tool]
            if tool.side_effect and not self.approve(tool.name, nxt.args):
                tr.violations.append(f"{tool.name} blocked: human approval missing")
                tr.steps.append(Step(nxt, {"error": "blocked: approval required"}, False))
                continue
            try:
                tr.steps.append(Step(nxt, tool.fn(**nxt.args), True))
            except Exception as e:  # tool errors become observations, never crashes
                tr.steps.append(Step(nxt, {"error": str(e)}, False))
        tr.answer = tr.answer or "Stopped: step budget exhausted."
        tr.violations.append("max_steps reached")
        return tr


Writing src/agentkit/agent.py


In [5]:
%%writefile src/agentkit/evaluate.py
"""Scenario-based evaluation: task success, exact tool-sequence accuracy, steps, guardrail events."""
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd

from .agent import Agent
from .tools import Outbox, make_tools

SCENARIOS = Path(__file__).parent / "resources" / "scenarios.jsonl"


def load_scenarios(path: str | Path = SCENARIOS) -> list[dict]:
    return [json.loads(line) for line in Path(path).read_text(encoding="utf-8").splitlines() if line.strip()]


def run_suite(planner_factory, scenarios: list[dict] | None = None, max_steps: int = 5) -> pd.DataFrame:
    rows = []
    for sc in scenarios or load_scenarios():
        tools = make_tools(Outbox())
        agent = Agent(planner_factory(tools), tools, max_steps=max_steps,
                      approve=(lambda t, a: True) if sc["approve"] else (lambda t, a: False))
        tr = agent.run(sc["query"])
        contains = all(s.lower() in tr.answer.lower() for s in sc["must_contain"])
        tools_ok = tr.tools_called == sc["expected_tools"]
        rows.append({"id": sc["id"], "query": sc["query"], "success": contains and tools_ok, "answer_ok": contains,
                     "tools_ok": tools_ok, "steps": len(tr.steps), "violations": len(tr.violations),
                     "tools": tr.tools_called, "answer": tr.answer})
    return pd.DataFrame(rows)


def summarize(df: pd.DataFrame) -> dict:
    return {"task_success": float(df["success"].mean()), "answer_ok": float(df["answer_ok"].mean()),
            "tool_sequence_acc": float(df["tools_ok"].mean()), "avg_steps": float(df["steps"].mean()),
            "guardrail_events": int(df["violations"].sum()), "n": int(len(df))}


Writing src/agentkit/evaluate.py


In [6]:
%%writefile src/agentkit/planners.py
"""Planners decide the next tool call or the final answer.

RulePlanner: CPU baseline = intent classifier (TF-IDF + logistic regression) + slot extraction by regex.
LLMPlanner: ReAct-style (Yao et al. 2022, arXiv:2210.03629) JSON tool calls from any `llm(prompt) -> str`.
"""
from __future__ import annotations

import json
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

from .agent import Action, Final, Trace

INTENT_EXAMPLES = {
    "fault_code": ["what does fault E102 mean", "explain error code E205", "alarm E330 on the belt", "code E101 meaning",
                   "what is fault code E401", "E210 popped up what should I do"],
    "procedure": ["how do I start the pump", "what is the vibration limit", "how often change compressor oil",
                  "belt tension check procedure", "what is lockout tagout", "max bearing temperature",
                  "when to clean the heat exchanger", "what oil for the compressor"],
    "calculation": ["compute 4000*3", "what is 7.1 - 4.5", "calculate 2 percent of 1200 which is 0.02*1200", "how much is 85+10"],
    "work_order": ["create a work order for E102", "open a ticket for fault E205", "raise a work order for pump P-200 due to E101",
                   "log a maintenance job for E330", "please create work order E401"],
    "out_of_scope": ["what is the weather today", "tell me a joke", "who won the football match", "book me a flight to Paris"],
}
CODE = re.compile(r"\b(E\d{3})\b", re.I)
EXPR = re.compile(r"([0-9\.\s\+\-\*/\(\)]{3,})")


class RulePlanner:
    def __init__(self):
        X, y = zip(*[(t, k) for k, v in INTENT_EXAMPLES.items() for t in v])
        self.clf = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True),
                                 LogisticRegression(max_iter=2000, C=10.0)).fit(X, y)

    def intent(self, q: str) -> str:
        return self.clf.predict([q])[0]

    def next(self, query: str, tr: Trace):
        intent = self.intent(query)
        done = tr.tools_called
        obs = [s.observation for s in tr.steps if s.ok]
        code = CODE.search(query)
        if code and intent in ("calculation", "procedure", "out_of_scope"):
            intent = "fault_code"  # slot evidence (a fault-code pattern) overrides a weak classifier guess
        if intent == "out_of_scope":
            return Final("This assistant only answers plant-maintenance questions.")
        if intent in ("fault_code", "work_order") and code is None:
            return Final("Please give the fault code (e.g. E102).")
        if intent == "fault_code":
            if "lookup_fault_code" not in done:
                return Action("lookup_fault_code", {"code": code.group(1).upper()})
            f = obs[-1]
            if "error" in f:
                return Final(f"Unknown fault code {f['code']}.")
            return Final(f"{f['code']} on {f['asset']}: {f['description']} (severity {f['severity']}). Action: {f['action']}",
                         [f"FAULT:{f['code']}"])
        if intent == "work_order":
            if "lookup_fault_code" not in done:
                return Action("lookup_fault_code", {"code": code.group(1).upper()})
            f = next(o for o in obs if isinstance(o, dict) and "code" in o)
            if "error" in f:
                return Final(f"Unknown fault code {f['code']}; no work order created.")
            if "create_work_order" not in done:
                return Action("create_work_order", {"asset": f["asset"], "priority": f["severity"],
                                                    "summary": f"{f['code']}: {f['description']}"})
            last = tr.steps[-1]
            if not last.ok:
                return Final(f"Work order NOT created ({last.observation['error']}). Suggested: {f['severity']} on {f['asset']}.")
            wo = last.observation
            return Final(f"Created {wo['id']} ({wo['priority']}) for {wo['asset']}: {wo['summary']}", [f"FAULT:{f['code']}"])
        if intent == "calculation":
            m = EXPR.search(query)
            if "calculator" not in done and m:
                return Action("calculator", {"expression": m.group(1).strip()})
            if obs and "result" in obs[-1]:
                return Final(f"Result: {obs[-1]['result']:g}")
            return Final("I could not find an arithmetic expression.")
        # procedure
        if "search_manual" not in done:
            return Action("search_manual", {"query": query})
        hits = obs[-1] if obs else []
        if not hits:
            return Final("I don't know — nothing relevant in the procedures.")
        return Final(hits[0]["text"], [h["id"] for h in hits[:1]])


REACT_PROMPT = """You are a plant-maintenance assistant. Use tools; never invent facts.
Tools:
{tools}
Reply with ONE JSON object per turn, either
{{"tool": "<name>", "args": {{...}}}}  or  {{"final": "<answer>", "sources": ["<ids>"]}}

Question: {query}
{history}
Next JSON:"""


class LLMPlanner:
    def __init__(self, llm, tools):
        self.llm, self.tools = llm, tools

    def next(self, query: str, tr: Trace):
        hist = "\n".join(f"Called {s.action.tool}({json.dumps(s.action.args)}) -> {json.dumps(s.observation)[:400]}" for s in tr.steps)
        desc = "\n".join(f"- {t.name}: {t.description}" for t in self.tools.values())
        raw = self.llm(REACT_PROMPT.format(tools=desc, query=query, history=hist))
        m = re.search(r"\{.*\}", raw, re.S)
        try:
            obj = json.loads(m.group(0)) if m else None
        except json.JSONDecodeError:
            obj = None
        if not isinstance(obj, dict):
            return Action("__invalid__", {"raw": raw[:200]})  # rejected by the guardrails, recorded as violation
        if "final" in obj:
            return Final(str(obj["final"]), [str(s) for s in obj.get("sources", [])])
        return Action(str(obj.get("tool")), obj.get("args") if isinstance(obj.get("args"), dict) else {})


Writing src/agentkit/planners.py


In [7]:
%%writefile src/agentkit/resources/kb.json
{
  "_note": "SYNTHETIC demo knowledge base (fictional ACME plant). Replace with the host's procedures in the PFE.",
  "passages": [
    {"id": "PROC-01", "title": "Pump P-200 start-up", "text": "Before starting pump P-200, open the suction valve fully, check that the casing is primed, then start the motor and open the discharge valve slowly within 30 seconds."},
    {"id": "PROC-02", "title": "Pump P-200 vibration limits", "text": "Normal vibration for pump P-200 is below 4.5 mm/s RMS. Between 4.5 and 7.1 mm/s schedule an inspection within 7 days. Above 7.1 mm/s stop the pump and inspect bearings."},
    {"id": "PROC-03", "title": "Compressor C-110 oil change", "text": "Change the oil of compressor C-110 every 4000 operating hours using ISO VG 46 oil. Drain while warm and replace the oil filter at each change."},
    {"id": "PROC-04", "title": "Conveyor B-7 belt tension", "text": "Check conveyor B-7 belt tension weekly. Belt sag must not exceed 2 percent of the span between idlers."},
    {"id": "PROC-05", "title": "Lockout-tagout", "text": "Before any maintenance, isolate all energy sources, apply a personal lock and tag, and verify zero energy by attempting a start."},
    {"id": "PROC-06", "title": "Bearing temperature", "text": "Bearing temperature on motors must stay below 85 degrees Celsius. Above 95 degrees Celsius trip the motor and inspect lubrication."},
    {"id": "PROC-07", "title": "Heat exchanger E-301 cleaning", "text": "Clean heat exchanger E-301 when the approach temperature rises by more than 5 degrees Celsius above the clean baseline."},
    {"id": "PROC-08", "title": "Work order priorities", "text": "Priority P1 means safety risk or production stop, fix within 4 hours. P2 means degraded operation, fix within 24 hours. P3 is planned maintenance within 7 days."}
  ],
  "fault_codes": {
    "E101": {"asset": "P-200", "description": "Suction pressure low", "severity": "P2", "action": "Check suction valve and strainer for blockage."},
    "E102": {"asset": "P-200", "description": "High vibration above 7.1 mm/s", "severity": "P1", "action": "Stop the pump and inspect bearings."},
    "E205": {"asset": "C-110", "description": "Oil pressure low", "severity": "P1", "action": "Stop the compressor and check oil level and pump."},
    "E210": {"asset": "C-110", "description": "Oil change due", "severity": "P3", "action": "Schedule oil change with ISO VG 46 oil."},
    "E330": {"asset": "B-7", "description": "Belt misalignment", "severity": "P2", "action": "Realign belt and check idlers."},
    "E401": {"asset": "E-301", "description": "Fouling suspected", "severity": "P3", "action": "Plan cleaning of heat exchanger."}
  }
}


Writing src/agentkit/resources/kb.json


In [8]:
%%writefile src/agentkit/resources/scenarios.jsonl
{"id": "s01", "query": "What does fault E102 mean?", "expected_tools": ["lookup_fault_code"], "must_contain": ["vibration", "P1"], "approve": false}
{"id": "s02", "query": "Explain error code E210 please", "expected_tools": ["lookup_fault_code"], "must_contain": ["Oil change", "P3"], "approve": false}
{"id": "s03", "query": "What is code E999?", "expected_tools": ["lookup_fault_code"], "must_contain": ["Unknown"], "approve": false}
{"id": "s04", "query": "What is the vibration limit for pump P-200 before I must stop it?", "expected_tools": ["search_manual"], "must_contain": ["7.1"], "approve": false}
{"id": "s05", "query": "How often should I change the compressor C-110 oil?", "expected_tools": ["search_manual"], "must_contain": ["4000"], "approve": false}
{"id": "s06", "query": "What is the maximum bearing temperature on motors?", "expected_tools": ["search_manual"], "must_contain": ["85"], "approve": false}
{"id": "s07", "query": "When must heat exchanger E-301 be cleaned?", "expected_tools": ["search_manual"], "must_contain": ["5 degrees"], "approve": false}
{"id": "s08", "query": "Compute 4000*3", "expected_tools": ["calculator"], "must_contain": ["12000"], "approve": false}
{"id": "s09", "query": "Create a work order for fault E205", "expected_tools": ["lookup_fault_code", "create_work_order"], "must_contain": ["WO-", "P1", "C-110"], "approve": true}
{"id": "s10", "query": "Open a ticket for E330", "expected_tools": ["lookup_fault_code", "create_work_order"], "must_contain": ["NOT created", "P2"], "approve": false}
{"id": "s11", "query": "Tell me a joke", "expected_tools": [], "must_contain": ["only answers"], "approve": false}
{"id": "s12", "query": "Create a work order for the pump", "expected_tools": [], "must_contain": ["fault code"], "approve": true}


Writing src/agentkit/resources/scenarios.jsonl


In [9]:
%%writefile src/agentkit/tools.py
"""Tools the agent may call. Each declares an argument schema and whether it has side effects."""
from __future__ import annotations

import ast
import json
import operator
import re
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Callable

import numpy as np
from rank_bm25 import BM25Okapi

KB = json.loads((Path(__file__).parent / "resources" / "kb.json").read_text(encoding="utf-8"))


def tokenize(t: str) -> list[str]:
    return re.findall(r"[a-z0-9\-\.]+", t.lower())


@dataclass
class Tool:
    name: str
    description: str
    args: dict[str, type]
    fn: Callable[..., Any]
    side_effect: bool = False


class Manual:
    def __init__(self, passages=None):
        self.p = passages or KB["passages"]
        self.bm25 = BM25Okapi([tokenize(x["title"] + " " + x["text"]) for x in self.p])

    def search(self, query: str, k: int = 2) -> list[dict]:
        s = self.bm25.get_scores(tokenize(query))
        return [{"id": self.p[i]["id"], "text": self.p[i]["text"]} for i in np.argsort(-s)[:k] if s[i] > 0]


_OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv,
        ast.Pow: operator.pow, ast.USub: operator.neg, ast.UAdd: operator.pos}


def safe_eval(expr: str) -> float:
    """Arithmetic only (+ - * / ** and parentheses). No names, calls or attributes."""
    def ev(n):
        if isinstance(n, ast.Expression):
            return ev(n.body)
        if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)):
            return n.value
        if isinstance(n, ast.BinOp) and type(n.op) in _OPS:
            return _OPS[type(n.op)](ev(n.left), ev(n.right))
        if isinstance(n, ast.UnaryOp) and type(n.op) in _OPS:
            return _OPS[type(n.op)](ev(n.operand))
        raise ValueError("unsupported expression")
    if len(expr) > 100:
        raise ValueError("expression too long")
    return float(ev(ast.parse(expr, mode="eval")))


@dataclass
class Outbox:
    work_orders: list[dict] = field(default_factory=list)


def make_tools(outbox: Outbox | None = None, manual: Manual | None = None) -> dict[str, Tool]:
    outbox = outbox if outbox is not None else Outbox()
    manual = manual or Manual()

    def lookup_fault_code(code: str) -> dict:
        c = KB["fault_codes"].get(code.strip().upper())
        return {"code": code.upper(), **c} if c else {"code": code.upper(), "error": "unknown fault code"}

    def create_work_order(asset: str, priority: str, summary: str) -> dict:
        if priority not in ("P1", "P2", "P3"):
            raise ValueError("priority must be P1, P2 or P3")
        wo = {"id": f"WO-{len(outbox.work_orders) + 1:04d}", "asset": asset, "priority": priority, "summary": summary}
        outbox.work_orders.append(wo)
        return wo

    tools = [
        Tool("search_manual", "Search maintenance procedures. Args: query (str).", {"query": str}, lambda query: manual.search(query)),
        Tool("lookup_fault_code", "Look up a fault code like E102. Args: code (str).", {"code": str}, lookup_fault_code),
        Tool("calculator", "Evaluate arithmetic, e.g. '4000*3'. Args: expression (str).", {"expression": str},
             lambda expression: {"result": safe_eval(expression)}),
        Tool("create_work_order", "Create a maintenance work order (SIDE EFFECT, needs human approval). "
             "Args: asset (str), priority (P1|P2|P3), summary (str).", {"asset": str, "priority": str, "summary": str},
             create_work_order, side_effect=True),
    ]
    return {t.name: t for t in tools}


Writing src/agentkit/tools.py


In [10]:
import pandas as pd
pd.set_option('display.max_colwidth', 90)
from agentkit.evaluate import run_suite, summarize
from agentkit.planners import RulePlanner
rule = RulePlanner()
df = run_suite(lambda tools: rule)
print(summarize(df))
df[['id', 'query', 'tools', 'answer', 'success']]

{'task_success': 1.0, 'answer_ok': 1.0, 'tool_sequence_acc': 1.0, 'avg_steps': 1.0, 'guardrail_events': 1, 'n': 12}


,id,query,tools,answer,success
0,s01,What does fault E102 mean?,[lookup_fault_code],E102 on P-200: High vibration above 7.1 mm/s (severity P1). Action: Stop the pump and ...,True
1,s02,Explain error code E210 please,[lookup_fault_code],E210 on C-110: Oil change due (severity P3). Action: Schedule oil change with ISO VG 4...,True
2,s03,What is code E999?,[lookup_fault_code],Unknown fault code E999.,True
3,s04,What is the vibration limit for pump P-200 before I must stop it?,[search_manual],Normal vibration for pump P-200 is below 4.5 mm/s RMS. Between 4.5 and 7.1 mm/s schedu...,True
4,s05,How often should I change the compressor C-110 oil?,[search_manual],Change the oil of compressor C-110 every 4000 operating hours using ISO VG 46 oil. Dra...,True
5,s06,What is the maximum bearing temperature on motors?,[search_manual],Bearing temperature on motors must stay below 85 degrees Celsius. Above 95 degrees Cel...,True
6,s07,When must heat exchanger E-301 be cleaned?,[search_manual],Clean heat exchanger E-301 when the approach temperature rises by more than 5 degrees ...,True
7,s08,Compute 4000*3,[calculator],Result: 12000,True
8,s09,Create a work order for fault E205,"[lookup_fault_code, create_work_order]",Created WO-0001 (P1) for C-110: E205: Oil pressure low,True
9,s10,Open a ticket for E330,"[lookup_fault_code, create_work_order]",Work order NOT created (blocked: approval required). Suggested: P2 on B-7.,True


### Guardrails in action (scripted LLM replies, no model needed)

In [11]:
from agentkit.agent import Agent
from agentkit.planners import LLMPlanner
from agentkit.tools import Outbox, make_tools
class Scripted:
    def __init__(self, r): self.r = list(r)
    def __call__(self, prompt): return self.r.pop(0)
out = Outbox(); tools = make_tools(out)
llm = Scripted(['{"tool": "delete_database", "args": {}}',
                '{"tool": "create_work_order", "args": {"asset": "P-200", "priority": "P1", "summary": "vibration"}}',
                '{"final": "Work order needs supervisor approval.", "sources": []}'])
tr = Agent(LLMPlanner(llm, tools), tools, approve=lambda t, a: False).run('Create a WO for P-200 vibration')
print('violations:', tr.violations); print('work orders created:', out.work_orders); print('answer:', tr.answer)

violations: ["tool 'delete_database' not allowed", 'create_work_order blocked: human approval missing']
work orders created: []
answer: Work order needs supervisor approval.


## [GPU — Colab T4] Open LLM as planner (ReAct-style JSON tool calls)
On a T4: `Qwen/Qwen2.5-1.5B-Instruct` on the full suite. On CPU: `Qwen/Qwen2.5-0.5B-Instruct` on 3 scenarios (smoke test, slow).
Expect a small model to score below the rule baseline — measuring that gap is exactly what the PFE evaluation harness is for.

In [12]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from agentkit.evaluate import load_scenarios
gpu = torch.cuda.is_available()
NAME = 'Qwen/Qwen2.5-1.5B-Instruct' if gpu else 'Qwen/Qwen2.5-0.5B-Instruct'
tok = AutoTokenizer.from_pretrained(NAME)
model = AutoModelForCausalLM.from_pretrained(NAME, torch_dtype=torch.float16 if gpu else torch.float32).to('cuda' if gpu else 'cpu')
def hf_llm(prompt):
    msgs = [{'role': 'user', 'content': prompt}]
    enc = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors='pt', return_dict=True).to(model.device)
    with torch.no_grad():
        outp = model.generate(**enc, max_new_tokens=96, do_sample=False)
    return tok.decode(outp[0, enc['input_ids'].shape[1]:], skip_special_tokens=True)
scen = load_scenarios() if gpu else [s for s in load_scenarios() if s['id'] in ('s01', 's04', 's11')]
llm_df = run_suite(lambda tools: LLMPlanner(hf_llm, tools), scen, max_steps=4)
print(NAME, summarize(llm_df))
llm_df[['id', 'tools', 'answer', 'success', 'violations']]

/usr/local/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights:   0%|          | 1/290 [00:01<06:23,  1.33s/it]

Loading weights:  63%|██████▎   | 183/290 [00:01<00:00, 164.73it/s]

Loading weights:  84%|████████▍ | 243/290 [00:01<00:00, 167.32it/s]

Loading weights:  98%|█████████▊| 284/290 [00:02<00:00, 170.11it/s]

Loading weights: 100%|██████████| 290/290 [00:02<00:00, 139.04it/s]

Qwen/Qwen2.5-0.5B-Instruct {'task_success': 0.0, 'answer_ok': 0.0, 'tool_sequence_acc': 0.0, 'avg_steps': 4.0, 'guardrail_events': 4, 'n': 3}


,id,tools,answer,success,violations
0,s01,"[__invalid__, lookup_fault_code, lookup_fault_code, lookup_fault_code]",Stopped: step budget exhausted.,False,2
1,s04,"[lookup_fault_code, lookup_fault_code, lookup_fault_code, lookup_fault_code]",Stopped: step budget exhausted.,False,1
2,s11,"[lookup_fault_code, lookup_fault_code, lookup_fault_code, lookup_fault_code]",Stopped: step budget exhausted.,False,1


## Next steps for the PFE
1. Connect real tools (CMMS/ticketing API in a sandbox, document search from proposal B) behind the same guardrails.
2. Grow the scenario suite with operators (100+ cases, incl. adversarial/unsafe requests); track success, tool accuracy, cost, latency per release.
3. Compare planners (rules, small open LLM, hosted LLM) on the same harness; add human-in-the-loop approval UI and audit log.